In [1]:
jsonl_path = "/content/thuvienphapluat_documents.jsonl"

In [2]:
import json
import random

def load_content_doc(jsonl_path):
    all_doc = []
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            doc = json.loads(line)
            content = doc.get("content")
            if content:
                all_doc.append(content)
    return all_doc

In [3]:
sampled_legal_docs = load_content_doc(jsonl_path)

raw_legal_docs = random.sample(sampled_legal_docs, min(50, len(sampled_legal_docs)))

In [4]:
raw_legal_docs_1 = load_content_doc(jsonl_path)

In [5]:
print(len(raw_legal_docs))

50


In [6]:
print(raw_legal_docs[5]) #72

CHÍNH PHỦ
-------
CỘNG HÒA XÃ HỘI
  CHỦ NGHĨA VIỆT NAM
Độc lập - Tự do - Hạnh phúc
---------------
Số: 52/NQ-CP
Hà Nội, ngày 18
  tháng 3 năm 2025
NGHỊ QUYẾT
VỀ
VIỆC PHÊ DUYỆT NHIỆM VỤ LẬP ĐIỀU CHỈNH QUY HOẠCH SỬ DỤNG ĐẤT QUỐC GIA
THỜI KỲ 2021 - 2030, TẦM NHÌN ĐẾN NĂM 2050
CHÍNH PHỦ
Căn cứ
Luật Tổ chức
Chính phủ
ngày 18 tháng 02 năm 2025;
Căn cứ
Luật Quy hoạch
ngày 24 tháng 11 năm 2017;
Căn cứ
Luật Đất
đai
ngày 18 tháng 01 năm 2024;
Căn cứ Luật sửa đổi, bổ sung một số điều của
Luật Đất đai số 31/2024/QH15
,
Luật Nhà ở số 27/2023/QH15
,
Luật Kinh doanh bất động sản số 29/2023QH15
và
Luật Các tổ chức tín dụng số 32/2024/QH15
(Luật
số
43/2024/QH15
ngày 29 tháng 6 năm 2024);
Căn cứ Nghị quyết số
174/2024/QH15
ngày 30 tháng 11 năm 2024 của Quốc
hội nước Cộng hòa xã hội chủ nghĩa Việt Nam khóa XV, Kỳ họp thứ 8;
Căn cứ Nghị định số
08/2022/NĐ-CP
ngày 10 tháng 01 năm 2022 của Chính phủ quy định chi tiết một số điều của
Luật Bảo vệ môi trường
;
Căn cứ Nghị định số
102/2024/NĐ-CP
ngày 30 tháng
7

In [7]:
from tqdm import tqdm
import re

def preprocess_text(docs):
    processed_docs = []

    DOC_TYPES = [
        "LUẬT", "NGHỊ ĐỊNH", "NGHỊ QUYẾT", "QUYẾT NGHỊ", "QUYẾT ĐỊNH",
        "THÔNG TƯ", "THÔNG TƯ LIÊN TỊCH", "PHÁP LỆNH", "LỆNH", "CHỈ THỊ",
        "CÔNG VĂN", "BIÊN BẢN", "HỢP ĐỒNG", "QUY CHẾ", "ĐIỀU LỆ", "THÔNG BÁO",
        "BÁO CÁO", "KẾ HOẠCH", "PHƯƠNG ÁN", "ĐỀ ÁN", "PHỤ LỤC", "DANH MỤC"
    ]

    LEADING_KEYWORDS = {
        "Căn cứ", "Theo đề nghị", "Chiểu theo", "Xét", "Xét đề nghị"
    }

    def is_full_upper(text):
        return text.isupper() and any(c.isalpha() for c in text)

    def normalize_doc_type(line):
        """
        Trả về DOC_TYPE nếu dòng bắt đầu bằng một DOC_TYPE hợp lệ.
        Với riêng 'LUẬT', yêu cầu cả dòng phải viết hoa toàn bộ (tránh nhầm 'Luật số').
        Các DOC_TYPE khác chỉ cần bắt đầu bằng từ đó (in hoa).
        """
        stripped_line = line.strip()

        for dt in DOC_TYPES:
            if dt == "LUẬT":
                if stripped_line == "LUẬT":
                    return "LUẬT"
            else:
                if stripped_line.upper().startswith(dt):
                    return dt

        return None

    def is_likely_leading_keyword(line):
        return any(line.startswith(k) for k in LEADING_KEYWORDS)

    def is_likely_structure_line(line):
        line = line.strip()
        return (
            re.match(r"^Điều\s+\d+", line) or
            re.match(r"^Chương\s+[IVXLC\d]+", line) or
            re.match(r"^Mục\s+[IVXLC\d]+", line) or
            re.match(r"^[IVXLC]+\.", line) or
            re.match(r"^\d+(\.\d+)*", line)
        )

    def is_separator(line):
        return re.fullmatch(r"[-*]{3,}", line.strip()) is not None

    for doc in tqdm(docs, desc="Processing documents"):
        lines = [line.strip() for line in doc.strip().splitlines()]
        result = []
        doc_type_count = 0
        stage = "search_doc_type"

        # === STEP 1: Tách theo 3 phần như yêu cầu ===
        sep_indices = [i for i, line in enumerate(lines) if is_separator(line)]

        part1, part2, part3 = [], [], []
        doc_type_start_idx = None

        if len(sep_indices) >= 2:
            i1, i2 = sep_indices[0], sep_indices[1]

            # part 1: trước dòng phân cách đầu tiên
            part1 = [l for l in lines[:i1] if l and not is_separator(l)]
            if part1:
                result.append(" ".join(part1))

            # part 2: giữa 2 dòng phân cách
            part2 = [l for l in lines[i1+1:i2] if l and not is_separator(l)]
            if part2:
                result.append(" ".join(part2))

            # part 3: sau phân cách thứ 2 đến trước dòng có DOC_TYPE
            for i, line in enumerate(lines[i2+1:], start=i2+1):
                if is_separator(line):
                    continue
                if normalize_doc_type(line.replace(":", "")):
                    doc_type_start_idx = i
                    break
                part3.append(line)

            if part3:
                result.append(" ".join(part3))

            lines = lines[doc_type_start_idx:] if doc_type_start_idx is not None else []
        else:
            # Nếu không đủ 2 separator, fallback: loại separator và gộp toàn bộ
            all_content = [l for l in lines if l and not is_separator(l)]
            if all_content:
                result.append(" ".join(all_content))
            lines = []  # không xử lý thêm

        # === STEP 2: Xử lý tiêu đề văn bản và phần căn cứ ===
        buffer = []
        i = 0
        while i < len(lines):
            raw_line = lines[i]
            line = raw_line.strip()
            if not line or is_separator(line):
                i += 1
                continue

            normalized = normalize_doc_type(line.replace(":", "").strip())

            if normalized:
                prev_line = lines[i - 1].strip() if i > 0 else ""
                next_line = lines[i + 1].strip() if i + 1 < len(lines) else ""

                is_continuation = is_likely_leading_keyword(prev_line)
                is_valid_doc_type_2 = is_likely_structure_line(next_line)

                if doc_type_count >= 1 and is_continuation:
                    buffer.append(line)
                    i += 1
                    continue

                if doc_type_count >= 1 and not is_valid_doc_type_2:
                    buffer.append(line)
                    i += 1
                    continue

                doc_type_count += 1

                if doc_type_count == 1 and is_full_upper(line):
                    result.append(line)
                    stage = "collect_upper"
                    buffer = []
                    i += 1
                    continue

                if doc_type_count == 2:
                    if buffer:
                        result.append(" ".join(buffer))
                        buffer = []
                    result.append(normalized)
                    i += 1

                    while i < len(lines):
                        tail_line = lines[i].strip()
                        if tail_line and not is_separator(tail_line):
                            result.append(tail_line)
                        i += 1
                    break

            if doc_type_count == 1:
                if stage == "collect_upper":
                    if is_full_upper(line):
                        buffer.append(line)
                        i += 1
                        continue
                    else:
                        if buffer:
                            result.append(" ".join(buffer))
                            buffer = []
                        stage = "collect_lower"
                        continue

                elif stage == "collect_lower":
                    buffer.append(line)
                    i += 1
                    continue

            result.append(line)
            i += 1

        if buffer:
            result.extend(buffer)

        processed_docs.append("\n".join(result))

    return processed_docs

In [8]:
preprocessed_legal_docs = preprocess_text(raw_legal_docs)

Processing documents: 100%|██████████| 50/50 [00:00<00:00, 249.99it/s]


In [9]:
print(preprocessed_legal_docs[2])

ỦY BAN NHÂN DÂN TỈNH BÌNH PHƯỚC
CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM Độc lập - Tự do - Hạnh phúc
Số: 1102/QĐ-UBND Bình Phước, ngày 12 tháng 7 năm 2024
QUYẾT ĐỊNH
CÔNG BỐ DANH MỤC CÁC THÀNH PHẦN HỒ SƠ THỦ TỤC HÀNH CHÍNH PHẢI THỰC HIỆN SỐ HÓA TRÊN ĐỊA BÀN TỈNH BÌNH PHƯỚC CHỦ TỊCH ỦY BAN NHÂN DÂN TỈNH
Căn cứ Luật Tổ chức Chính quyền địa phương ngày 19/6/2015; Căn cứ Luật sửa đổi, bổ sung một số điều của Luật Tổ chức Chính phủ và Luật Tổ chức Chính quyền địa phương ngày 22/11/2019; Căn cứ Nghị định số 63/2010/NĐ-CP ngày 08/6/2010 của Chính phủ về kiểm soát thủ tục hành chính; Nghị định số 92/2017/NĐ-CP ngày 07/8/2017 của Chính phủ sửa đổi, bổ sung một số điều của các Nghị định liên quan đến kiểm soát thủ tục hành chính; Căn cứ Nghị định số 61/2018/NĐ-CP ngày 23/4/2018 của Chính phủ về thực hiện cơ chế một cửa, một cửa liên thông trong giải quyết thủ tục hành chính; Nghị định số 107/2021/NĐ-CP ngày 06/12/2021 về việc sửa đổi, bổ sung một số điều của Nghị định số 61/2018/NĐ-CP ngày 23/4/201

In [10]:
import re

def preprocess_legal_documents_to_markdown(docs):
    processed_docs = []

    DOC_TYPES = [
        "LUẬT", "NGHỊ ĐỊNH", "NGHỊ QUYẾT", "QUYẾT NGHỊ", "QUYẾT ĐỊNH",
        "THÔNG TƯ", "THÔNG TƯ LIÊN TỊCH", "PHÁP LỆNH", "LỆNH", "CHỈ THỊ",
        "CÔNG VĂN", "BIÊN BẢN", "HỢP ĐỒNG", "QUY CHẾ", "ĐIỀU LỆ",
        "THÔNG BÁO", "BÁO CÁO", "KẾ HOẠCH", "PHƯƠNG ÁN", "ĐỀ ÁN",
        "PHỤ LỤC", "DANH MỤC"
    ]

    for doc in docs:
        lines = doc.strip().splitlines()
        result = []

        i = 0
        doc_type_count = 0

        # === Biến trạng thái ===
        inside_dieu = False
        last_number_heading_allowed = False

        while i < len(lines):
            line = lines[i].strip()
            if not line:
                i += 1
                continue

            matched_doc_type = None
            if line == line.upper():
                for dt in DOC_TYPES:
                    if line.startswith(dt):
                        matched_doc_type = dt
                        break

            if matched_doc_type:
                doc_type_count += 1
                heading_level = "#" if doc_type_count == 1 else "##"
                result.append(f"{heading_level} {matched_doc_type.title()}")
                inside_dieu = False
                last_number_heading_allowed = False
                i += 1
                continue

            # 1. Chương
            m = re.match(r"^(Chương\s+[IVXLC\d]+)", line, re.IGNORECASE)
            if m:
                result.append(f"### {m.group(1)}")
                rest = line[m.end():].strip()
                if rest:
                    result.append(rest)
                inside_dieu = False
                last_number_heading_allowed = False
                i += 1
                continue

            # 2. Mục
            m = re.match(r"^(Mục\s+[IVXLC\d]+)", line, re.IGNORECASE)
            if m:
                result.append(f"#### {m.group(1)}")
                rest = line[m.end():].strip()
                if rest:
                    result.append(rest)
                inside_dieu = False
                last_number_heading_allowed = False
                i += 1
                continue

            # 3. Điều
            m = re.match(r"^(Điều\s+\d+\.)", line)
            if m:
                result.append(f"##### {m.group(1)}")
                rest = line[m.end():].strip()
                if rest:
                    result.append(rest)
                inside_dieu = True  # Đang trong một điều
                last_number_heading_allowed = False
                i += 1
                continue

            # 4. I., II., III.
            m = re.match(r"^([IVXLC]+)\.\s*(.*)", line)
            if m:
                result.append(f"### {m.group(1)}.")
                if m.group(2):
                    result.append(m.group(2))
                inside_dieu = False
                last_number_heading_allowed = False
                i += 1
                continue

            # 5. Dạng "1." không kèm nội dung
            if re.match(r"^(?!202\d)([1-9][0-9]{0,2})\.\s*$", line):
                if not inside_dieu:
                    result.append(f"#### {line.strip()}")
                    last_number_heading_allowed = True
                else:
                    result.append(line)  # không markdown nếu trong Điều
                    last_number_heading_allowed = False
                i += 1
                continue

            # 6. Dạng "1. Nội dung"
            m = re.match(r"^(?!202\d)([1-9][0-9]{0,2})\.\s+(.+)", line)
            if m:
                if not inside_dieu:
                    result.append(f"#### {m.group(1)}.")
                    result.append(m.group(2))
                    last_number_heading_allowed = True
                else:
                    result.append(line)
                    last_number_heading_allowed = False
                i += 1
                continue

            # 7. Dạng "1.1 Nội dung"
            m = re.match(r"^([1-9]\d*(\.[1-9]\d*){1,2})\s+(.+)", line)
            if m:
                if last_number_heading_allowed:
                    result.append(f"##### {m.group(1)}")
                    result.append(m.group(3))
                else:
                    result.append(line)  # không markdown nếu số cha không được markdown
                i += 1
                continue

            # Nội dung thông thường
            result.append(line)
            i += 1

        processed_docs.append("\n".join(result))

    return processed_docs

In [11]:
markdown_legal_docs = preprocess_legal_documents_to_markdown(preprocessed_legal_docs)

In [12]:
print(markdown_legal_docs[6]) #136

BỘ KHOA HỌC VÀ CÔNG NGHỆ
CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM Độc lập - Tự do - Hạnh phúc
Số: 2324/QĐ-BKHCN Hà Nội, ngày 13 tháng 09 năm 2024
# Quyết Định
BAN HÀNH DANH MỤC THÀNH PHẦN HỒ SƠ PHẢI SỐ HÓA ĐỐI VỚI THỦ TỤC HÀNH CHÍNH THUỘC PHẠM VI, CHỨC NĂNG QUẢN LÝ CỦA BỘ KHOA HỌC VÀ CÔNG NGHỆ BỘ TRƯỞNG BỘ KHOA HỌC VÀ CÔNG NGHỆ
Căn cứ Nghị định số 28/2023/NĐ-CP ngày 02/6/2023 của Chính phủ quy định chức năng, nhiệm vụ, quyền hạn và cơ cấu tổ chức của Bộ Khoa học và Công nghệ; Căn cứ Nghị định số 63/2010/NĐ-CP ngày 08/6/2010 của Chính phủ về kiểm soát thủ tục hành chính; Nghị định số 48/2013/NĐ-CP ngày 14/5/2013 và Nghị định số 92/2017/NĐ-CP ngày 07/8/2017 của Chính phủ sửa đổi, bổ sung một số điều của các nghị định liên quan đến kiểm soát thủ tục hành chính; Căn cứ Nghị định số 45/2020/NĐ-CP ngày 08/4/2020 của Chính phủ quy định về thực hiện thủ tục hành chính trên môi trường điện tử; Căn cứ Quyết định số 104/QĐ-TTg ngày 25/01/2024 của Thủ tướng Chính phủ ban hành kế hoạch cải cách thủ tục h

# 3. Metadata Enrichment

In [13]:
import re

def extract_legal_metadata(docs):
    """
    Trích xuất metadata từ danh sách các văn bản pháp lý.
    Trả về danh sách dict chứa metadata tương ứng từng văn bản.
    """

    all_metadata = []

    for doc in docs:
        lines = doc.strip().splitlines()
        lines = [line.strip() for line in lines if line.strip() != ""]

        metadata = {
            "co_quan_ban_hanh": "",
            "ma_so": "",
            "ngay_ban_hanh": "",
            "noi_ban_hanh": "",
            "loai_van_ban": "",
            "chu_de": ""
        }

        # 1. Cơ quan ban hành: dòng đầu tiên
        if len(lines) >= 1:
            metadata["co_quan_ban_hanh"] = lines[0].lower()

        # 2. Dòng thứ 3: chứa mã số và ngày ban hành
        if len(lines) >= 3:
            line3 = lines[2].strip()

            # Tách riêng mã số
            ma_so_match = re.search(r"Số\s*:\s*([\w\/\-]+)", line3, re.IGNORECASE)
            if ma_so_match:
                metadata["ma_so"] = ma_so_match.group(1).strip()

                # Loại bỏ phần "Số: xxx" để tránh dính vào nơi ban hành
                line3 = re.sub(r"Số\s*:\s*[\w\/\-]+", "", line3, flags=re.IGNORECASE).strip(" ,")

                # Loại bỏ từ mở đầu như "Luật", "Nghị định", v.v nếu có
                line3 = re.sub(r"^\b(luật|nghị[^\s]*)\b", "", line3, flags=re.IGNORECASE).strip()


            # Tách nơi ban hành và ngày
            date_match = re.search(
                r"(.*?),?\s*ngày\s+(\d{1,2})\s+tháng\s+(\d{1,2})\s+năm\s+(\d{4})",
                line3,
                re.IGNORECASE
            )
            if date_match:
                place = date_match.group(1).strip()
                day = int(date_match.group(2))
                month = int(date_match.group(3))
                year = int(date_match.group(4))

                metadata["noi_ban_hanh"] = place.title()
                metadata["ngay_ban_hanh"] = f"{day:02d}/{month:02d}/{year}"

        # 3. Loại văn bản: dòng thứ 4
        if len(lines) >= 4:
            loai = lines[3].strip()
            loai = re.sub(r"^#+\s*", "", loai)
            metadata["loai_van_ban"] = loai.title()

        # 4. Chủ đề: dòng thứ 5
        if len(lines) >= 5:
            metadata["chu_de"] = lines[4].strip().capitalize()

        all_metadata.append(metadata)

    return all_metadata

In [14]:
legal_metadata = extract_legal_metadata(markdown_legal_docs)

In [15]:
print(legal_metadata[6])

{'co_quan_ban_hanh': 'bộ khoa học và công nghệ', 'ma_so': '2324/QĐ-BKHCN', 'ngay_ban_hanh': '13/09/2024', 'noi_ban_hanh': 'Hà Nội', 'loai_van_ban': 'Quyết Định', 'chu_de': 'Ban hành danh mục thành phần hồ sơ phải số hóa đối với thủ tục hành chính thuộc phạm vi, chức năng quản lý của bộ khoa học và công nghệ bộ trưởng bộ khoa học và công nghệ'}


In [16]:
metadata_label_map = {
    "dieu": "Điều",
    "muc" : "Mục",
    "chuong": "Chương",
    "loai_van_ban": "Loại văn bản",
    "chu_de": "Chủ đề",
    "ma_so": "Mã số",
    "ngay_ban_hanh": "Ngày ban hành",
    "noi_ban_hanh": "Nơi ban hành",
    "co_quan_ban_hanh": "Cơ quan ban hành",
    "can_cu" : "Căn cứ"
}

# 4. Indexing

In [17]:
!pip install -U langchain-openai langchain-chroma langchain-community langchain-huggingface --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 86.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 4.5 MB/s eta 0:00:00


In [18]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.docstore.document import Document
from langchain.text_splitter import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

In [20]:
embedding_model = HuggingFaceEmbeddings(
    model_name="AITeamVN/Vietnamese_Embedding",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

In [21]:
def clean_metadata(metadata):
    cleaned = {}
    for key, value in metadata.items():
        if not value:
            cleaned[key] = value
            continue

        value = str(value).strip()

        if key in {"chuong", "muc"}:
            match = re.search(r"(?:Chương|Mục)?\s*([IVXLCDM\d]+)", value, re.IGNORECASE)
            cleaned[key] = match.group(1).upper() if match else value

        elif key == "dieu":
            match = re.search(r"(\d+(?:\.\d+)*)(?:\.*)?", value)
            cleaned[key] = match.group(1) if match else value.lower()

        elif key == "ma_so":
            cleaned[key] = value

        else:
            cleaned[key] = value.lower()

    return cleaned

In [22]:
def create_vectordb(
    docs,
    legal_metadata: list[dict],
    embedding_model,
    persist_directory: str,
    headers_to_split_on=None,
    collection_name="law_docs"
):
    if headers_to_split_on is None:
        headers_to_split_on = [
            ("#", "chu_de"),
            ("##", "can_cu"),
            ("###", "chuong"),
            ("####", "muc"),
            ("#####", "dieu")
        ]

    parent_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=headers_to_split_on,
        strip_headers=True
    )

    child_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1024,
        chunk_overlap=100
    )

    parent_documents = []

    for i, raw_doc in enumerate(tqdm(docs, desc="Tạo parent documents")):
        parent_chunks = parent_splitter.split_text(raw_doc)
        base_metadata = legal_metadata[i] if i < len(legal_metadata) else {}

        for chunk in parent_chunks:
            merged_metadata = {
                **chunk.metadata,
                **base_metadata,
                "sourcedoc": f"doc{i}"
            }

            cleaned_metadata = clean_metadata(merged_metadata)

            parent_documents.append(Document(
                page_content=chunk.page_content,
                metadata=cleaned_metadata
            ))

    if not parent_documents:
        raise ValueError("Không có document nào được tạo ra.")

    vectorstore = Chroma(
        collection_name=collection_name,
        embedding_function=embedding_model,
        persist_directory=persist_directory
    )

    child_documents = []
    for i, parent_doc in enumerate(tqdm(parent_documents, desc="Tạo child documents")):
        parent_id = f"parent-{i}"
        parent_doc.metadata["parent_id"] = parent_id

        chunks = child_splitter.create_documents(
            [parent_doc.page_content],
            metadatas=[{
                "parent_id": parent_id,
                **parent_doc.metadata
            }]
        )
        child_documents.extend(chunks)

    batch_size = 500
    for i in tqdm(range(0, len(child_documents), batch_size), desc="Lưu vectorstore"):
        vectorstore.add_documents(child_documents[i:i + batch_size])

    vectorstore.persist()
    return parent_documents

In [23]:
def get_retriever(
    parent_documents,
    embedding_model,
    persist_directory: str,
    collection_name="law_docs",
    chunk_size=1024,
    chunk_overlap=100
):
    vectorstore = Chroma(
        collection_name=collection_name,
        embedding_function=embedding_model,
        persist_directory=persist_directory
    )

    docstore = InMemoryStore()
    store_items = []
    child_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    child_documents = []

    for i, parent_doc in enumerate(tqdm(parent_documents, desc="Tạo docstore & child docs")):
        parent_id = parent_doc.metadata["parent_id"]
        store_items.append((parent_id, parent_doc))

        chunks = child_splitter.create_documents(
            [parent_doc.page_content],
            metadatas=[{
                "parent_id": parent_id,
                **parent_doc.metadata
            }]
        )
        child_documents.extend(chunks)

    docstore.mset(store_items)

    retriever = ParentDocumentRetriever(
        vectorstore=vectorstore,
        docstore=docstore,
        child_splitter=child_splitter
    )

    return retriever, child_documents

In [24]:
parent_docs = create_vectordb(
    docs=markdown_legal_docs,
    legal_metadata=legal_metadata,
    embedding_model=embedding_model,
    persist_directory="./vector_db"
)

Tạo parent documents: 100%|██████████| 50/50 [00:00<00:00, 226.25it/s]
/tmp/ipython-input-22-3781990007.py:51: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(
Lưu vectorstore:   0%|          | 0/5 [00:52<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
retriever, child_docs = get_retriever(
    parent_documents=parent_docs,
    embedding_model=embedding_model,
    persist_directory="./vector_db"
)

In [34]:
print(len(parent_docs))
print(len(child_docs))

3125
4296


In [36]:
print(parent_docs[32])

page_content='Bộ Tư pháp chủ trì, phối
hợp với các cơ quan, địa phương tiếp tục rà soát các quy định pháp luật để kịp
thời phát hiện, tháo gỡ các hạn chế, bất cập trong cơ chế, chính sách, pháp luật
theo thẩm quyền, kịp thời báo cáo Chính phủ, Thủ tướng Chính phủ đối với các vấn
đề vượt thẩm quyền.' metadata={'chu_de': 'phiên họp chính phủ thường kỳ tháng 10 năm 2023 chính phủ', 'can_cu': 'phụ lục', 'muc': '16', 'co_quan_ban_hanh': 'chính phủ', 'ma_so': '185/NQ-CP', 'ngay_ban_hanh': '07/11/2023', 'noi_ban_hanh': 'hà nội', 'loai_van_ban': 'nghị quyết', 'sourcedoc': 'doc0', 'parent_id': 'parent-32'}


In [37]:
print(child_docs[32])

page_content='Các cơ quan chủ chương trình, cơ quan chủ
trì dự án thành phần chủ động xây dựng các văn bản hướng dẫn việc triển khai thực
hiện các chương trình mục tiêu quốc gia năm 2024 để chủ động, kịp thời hướng dẫn
các địa phương ngay sau khi kế hoạch vốn ngân sách trung ương thực hiện các
chương trình mục tiêu quốc gia được cấp có thẩm quyền giao. Các địa phương căn
cứ chức năng, nhiệm vụ, quyền hạn, chủ động hướng dẫn, chỉ đạo thực hiện phù hợp,
hiệu quả.' metadata={'parent_id': 'parent-14', 'chu_de': 'phiên họp chính phủ thường kỳ tháng 10 năm 2023 chính phủ', 'can_cu': 'quyết nghị', 'chuong': 'IV', 'muc': '4', 'co_quan_ban_hanh': 'chính phủ', 'ma_so': '185/NQ-CP', 'ngay_ban_hanh': '07/11/2023', 'noi_ban_hanh': 'hà nội', 'loai_van_ban': 'nghị quyết', 'sourcedoc': 'doc0'}


In [72]:
for doc in child_docs:
    if doc.metadata.get("sourcedoc") == "doc5":
        print("\n=== Parent Document ===")
        print("Metadata:", doc.metadata)
        print("Content preview:", doc.page_content[:100])  # In 200 ký tự đầu nội dung


=== Parent Document ===
Metadata: {'parent_id': 'parent-146', 'co_quan_ban_hanh': 'chính phủ', 'ma_so': '33/NQ-CP', 'ngay_ban_hanh': '12/02/2025', 'noi_ban_hanh': 'hà nội', 'loai_van_ban': 'nghị quyết', 'chu_de': 'về trình dự thảo nghị quyết của quốc hội về thí điểm một số chính sách để tháo gỡ vướng mắc trong hoạt động khoa học, công nghệ, đổi mới sáng tạo và chuyển đổi số quốc gia chính phủ', 'sourcedoc': 'doc5'}
Content preview: CHÍNH PHỦ
CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM Độc lập - Tự do - Hạnh phúc
Số: 33/NQ-CP Hà Nội, ngày 1

=== Parent Document ===
Metadata: {'parent_id': 'parent-147', 'chu_de': 'về trình dự thảo nghị quyết của quốc hội về thí điểm một số chính sách để tháo gỡ vướng mắc trong hoạt động khoa học, công nghệ, đổi mới sáng tạo và chuyển đổi số quốc gia chính phủ', 'co_quan_ban_hanh': 'chính phủ', 'ma_so': '33/NQ-CP', 'ngay_ban_hanh': '12/02/2025', 'noi_ban_hanh': 'hà nội', 'loai_van_ban': 'nghị quyết', 'sourcedoc': 'doc5'}
Content preview: VỀ TRÌNH DỰ THẢO NGHỊ QUY

In [38]:
# Kiểm tra retriever có hoạt động không
print("Kiểm tra retriever:")
print(f"Type: {type(retriever)}")
print(f"Vectorstore type: {type(retriever.vectorstore)}")
print(f"Docstore type: {type(retriever.docstore)}")

# Kiểm tra số lượng documents
try:
    doc_count = retriever.vectorstore._collection.count()
    print(f"Số documents trong vectorstore: {doc_count}")
except Exception as e:
    print(f"Lỗi khi đếm documents: {e}")

# Kiểm tra docstore
print(f"Số parent documents: {len(retriever.docstore.store)}")

Kiểm tra retriever:
Type: <class 'langchain.retrievers.parent_document_retriever.ParentDocumentRetriever'>
Vectorstore type: <class 'langchain_community.vectorstores.chroma.Chroma'>
Docstore type: <class 'langchain_core.stores.InMemoryStore'>
Số documents trong vectorstore: 4296
Số parent documents: 3125


In [ ]:
query = "Các bộ, cơ quan, địa phương căn cứ chức năng, nhiệm vụ, quyền hạn được giao như thế nào trong nghị quyết Số: 156/NQ-CP ?"

print(f"Query: {query}")
print(f"Query length: {len(query)}")

# Thử các cách khác nhau
try:
    # Cách 1: Sử dụng invoke
    results1 = retriever.invoke(query)
    print(f"Invoke results: {len(results1) if results1 else 0}")

    # Cách 2: Sử dụng get_relevant_documents
    results2 = retriever.get_relevant_documents(query)
    print(f"Get_relevant_documents results: {len(results2) if results2 else 0}")

    # Cách 3: Trực tiếp từ vectorstore
    results3 = retriever.vectorstore.similarity_search(query, k=5)
    print(f"Direct vectorstore results: {len(results3) if results3 else 0}")

except Exception as e:
    print(f"Lỗi khi search: {e}")

Query: Các bộ, cơ quan, địa phương căn cứ chức năng, nhiệm vụ, quyền hạn được giao như thế nào trong nghị quyết Số: 156/NQ-CP ?
Query length: 120
Invoke results: 0
Get_relevant_documents results: 0
Direct vectorstore results: 5


/tmp/ipython-input-26-2533028128.py:13: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  results2 = retriever.get_relevant_documents(query)


In [ ]:
query = "Các bộ, cơ quan, địa phương căn cứ chức năng, nhiệm vụ, quyền hạn được giao như thế nào trong nghị quyết Số: 156/NQ-CP ?"

print("=== DEBUGGING PARENT DOCUMENT RETRIEVER ===")

# 1. Kiểm tra child documents từ vectorstore
child_docs = retriever.vectorstore.similarity_search(query, k=5)
print(f"Child documents found: {len(child_docs)}")

for i, doc in enumerate(child_docs):
    print(f"\n--- Child Doc {i+1} ---")
    print(f"Content: {doc.page_content[:200]}...")
    print(f"Metadata: {doc.metadata}")
    parent_id = doc.metadata.get('parent_id')
    print(f"Parent ID: {parent_id}")

    # 2. Kiểm tra parent document có tồn tại không
    if parent_id:
        parent_doc = retriever.docstore.store.get(parent_id)
        if parent_doc:
            print(f"✓ Parent document exists")
            print(f"Parent content: {parent_doc.page_content[:100]}...")
        else:
            print(f"✗ Parent document NOT FOUND in docstore!")
    else:
        print(f"✗ No parent_id in metadata!")

# 3. Kiểm tra docstore
print(f"\n=== DOCSTORE INFO ===")
print(f"Total items in docstore: {len(retriever.docstore.store)}")
print(f"Docstore keys: {list(retriever.docstore.store.keys())[:10]}...")  # In 10 keys đầu

# 4. Kiểm tra mapping parent_id
parent_ids_in_children = set()
for doc in child_docs:
    parent_id = doc.metadata.get('parent_id')
    if parent_id:
        parent_ids_in_children.add(parent_id)

print(f"Parent IDs in child documents: {parent_ids_in_children}")
print(f"Parent IDs in docstore: {set(retriever.docstore.store.keys())}")

=== DEBUGGING PARENT DOCUMENT RETRIEVER ===
Child documents found: 5

--- Child Doc 1 ---
Content: Về nhiệm vụ cụ thể của
các bộ, cơ quan, địa phương: Từng bộ, cơ quan, địa phương theo chức năng, nhiệm
vụ, quyền hạn được giao khẩn trương tổ chức triển khai thực hiện các nội dung tại
Phụ lục ban hàn...
Metadata: {'muc': '10', 'can_cu': 'quyết nghị', 'ngay_ban_hanh': '10/12/2024', 'parent_id': 'parent-870', 'chuong': 'I', 'co_quan_ban_hanh': 'chính phủ', 'loai_van_ban': 'nghị quyết', 'noi_ban_hanh': 'hà nội', 'source_doc': 'doc_37', 'ma_so': '233/NQ-CP', 'chu_de': 'phiên họp chính phủ thường kỳ tháng 11 năm 2024 chính phủ'}
Parent ID: parent-870
✓ Parent document exists
Parent content: Về nhiệm vụ cụ thể của
các bộ, cơ quan, địa phương: Từng bộ, cơ quan, địa phương theo chức năng, nhi...

--- Child Doc 2 ---
Content: Các sở, ban, ngành; UBND các huyện,
thị xã, thành phố, các đơn vị liên quan căn cứ nhiệm vụ được giao lập dự toán
ngân sách hằng năm lồng ghép với thực hiện nhiệm vụ của đơn

In [ ]:
query = "Các bộ, cơ quan, địa phương căn cứ chức năng, nhiệm vụ, quyền hạn được giao như thế nào trong nghị quyết Số: 156/NQ-CP ?"

child_docs = retriever.vectorstore.similarity_search(query, k = 20)
for doc in child_docs:
    print("\n--- Kết quả ---")
    print("Nội dung:", doc.page_content)
    print("Metadata:", doc.metadata)


--- Kết quả ---
Nội dung: Về nhiệm vụ cụ thể của
các bộ, cơ quan, địa phương: Từng bộ, cơ quan, địa phương theo chức năng, nhiệm
vụ, quyền hạn được giao khẩn trương tổ chức triển khai thực hiện các nội dung tại
Phụ lục ban hành kèm theo Nghị quyết này.
Metadata: {'chuong': 'I', 'chu_de': 'phiên họp chính phủ thường kỳ tháng 11 năm 2024 chính phủ', 'noi_ban_hanh': 'hà nội', 'parent_id': 'parent-870', 'ngay_ban_hanh': '10/12/2024', 'muc': '10', 'source_doc': 'doc_37', 'can_cu': 'quyết nghị', 'ma_so': '233/NQ-CP', 'loai_van_ban': 'nghị quyết', 'co_quan_ban_hanh': 'chính phủ'}

--- Kết quả ---
Nội dung: Các sở, ban, ngành; UBND các huyện,
thị xã, thành phố, các đơn vị liên quan căn cứ nhiệm vụ được giao lập dự toán
ngân sách hằng năm lồng ghép với thực hiện nhiệm vụ của đơn vị, địa phương gửi
cơ quan tài chính cùng cấp để bố trí kinh phí thực hiện theo quy định.
Metadata: {'muc': '2', 'source_doc': 'doc_3', 'chuong': 'V', 'ma_so': '39/KH-UBND', 'ngay_ban_hanh': '04/03/2022', 'noi_ban_hanh

In [50]:
def get_context(query):

    def format_metadata_vietnamese(metadata):
        parts = []
        for key, value in metadata.items():
            if key in ("sourcedoc", "parent_id"):
                label = metadata_label_map.get(key, key)
                continue

            label = metadata_label_map.get(key, key)
            parts.append(f"{label}: {value}")

        return "(Metadata: " + "; ".join(parts) + ")"



    child_docs = retriever.vectorstore.similarity_search(query, k=5)

    unique_parent_docs = {}
    for child_doc in child_docs:
        parent_id = child_doc.metadata.get("parent_id")
        if parent_id and parent_id not in unique_parent_docs:
            parent_doc = retriever.docstore.mget([parent_id])[0]
            if parent_doc:
                unique_parent_docs[parent_id] = parent_doc

    parent_docs = list(unique_parent_docs.values())

    contexts = []
    for i, doc in enumerate(parent_docs, 1):
        content = doc.page_content
        metadata_formatted = format_metadata_vietnamese(doc.metadata)
        contexts.append(f"Tài liệu {i}:\n{content}\n{metadata_formatted}")

    context = "\n\n".join(contexts)

    return context

In [64]:
query = "Nghị quyết Số: 33/NQ-CP gồm những nội dung gì?"

context = get_context(query)
print(context)

Tài liệu 1:
CHÍNH PHỦ
CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM Độc lập - Tự do - Hạnh phúc
Số: 33/NQ-CP Hà Nội, ngày 12 tháng 02 năm 2025
(Metadata: Cơ quan ban hành: chính phủ; Mã số: 33/NQ-CP; Ngày ban hành: 12/02/2025; Nơi ban hành: hà nội; Loại văn bản: nghị quyết; Chủ đề: về trình dự thảo nghị quyết của quốc hội về thí điểm một số chính sách để tháo gỡ vướng mắc trong hoạt động khoa học, công nghệ, đổi mới sáng tạo và chuyển đổi số quốc gia chính phủ)

Tài liệu 2:
CHÍNH PHỦ
CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM Độc lập - Tự do - Hạnh phúc
Số: 135/NQ-CP Hà Nội, ngày 30 tháng 8 năm 2023
(Metadata: Cơ quan ban hành: chính phủ; Mã số: 135/NQ-CP; Ngày ban hành: 30/08/2023; Nơi ban hành: hà nội; Loại văn bản: nghị quyết; Chủ đề: phiên họp chuyên đề về xây dựng pháp luật tháng 8 năm 2023 chính phủ)

Tài liệu 3:
115/NQ-CP
ngày 28 tháng 7 năm 2023; Nghị quyết số
135/NQ-CP
ngày 30 tháng 8 năm 2023.
2
Khoản 2 Điều 14
và Điều 19 Hiến pháp năm 2013 quy định quyền con người, quyền công dân chỉ có
thể bị

In [ ]:
persist_dir = "./vector_db_2"

law_db = Chroma(
    collection_name="law_docs",
    embedding_function=embedding_model,
    persist_directory=persist_dir
)

system_prompt = """Bạn là một chuyên gia tư vấn pháp luật với 30 năm kinh nghiệm trong lĩnh vực này.
Nhiệm vụ của bạn là trả lời câu hỏi của người dùng về pháp luật của Việt Nam dựa trên thông tin được cung cấp.
Bạn phải đảm bảo trả lời câu hỏi một các chính xác, đầy đủ, dễ hiểu và phù hợp với các quy định pháp luật hiện hành.

**Yêu cầu trả lời:**

### 1. Phân tích câu hỏi:
- Hiểu rõ toàn diện về câu hỏi, bao gồm các cách diễn đạt tương tự, các từ đồng nghĩa và các biên thể ngữ nghĩa hoặc cách diễn đạt không trực tiếp.
- Xác định rõ các khía cạnh chính hoặc các điểm cần làm rõ từ câu hỏi

### 2. Cấu trúc câu trả lời:
- Mở đầu bằng một câu trả lời chính cho câu hỏi một cách tổng hợp tóm tắt, ngắn gọn, bao quát đầy đủ ý, đi thẳng vào trọng tâm vấn đề chính.
- Trả lời theo từng ý rõ ràng, mỗi ý tương ứng với một điểm hoặc khía cạnh cụ thể.
- Sử dụng các số thứ tự (1, 2, 3...) hoặc dấu đầu dòng để trình bày các ý của nội dung một cách logic .
- Lưu ý bắt buộc: Với **mỗi ý** trong câu trả lời, bạn **bắt buộc phải trích dẫn chính xác nguồn từ phần Metadata** của tài liệu tham khảo tương ứng khi dùng để trả lời ý đó.

### 3. **Trích dẫn nguồn:**
- Trích dẫn nuồn theo mẫu sau: (*Nguồn: Điều [Điều] [Điều] - Mục [Mục] - Chương [Chương], văn bản [Loại văn bản]: [Chủ đề], số [Mã số], ban hành ngày [Ngày ban hành], tại [Nơi ban hành], bởi [Cơ quan ban hành]).
- Các ví dụ mẫu:
(*Nguồn: Điều 14 - Mục 3 - Chương II, văn bản Nghị quyết: Phiên họp chuyên đề về xây dựng pháp luật tháng 8 năm 2023 chính phủ, số 64/2025/QH15, ban hành ngày 19/02/2025 tại Hà Nội, bởi Quốc hội)
(*Nguồn: Điều 47, văn bản Luật: Ban hành văn bản quy phạm pháp luật, số 64/2025/QH15, ban hành ngày 19/02/2025 tại Hà Nội, bởi Quốc hội)
(*Nguồn: Điều 4 - Chương I, văn bản Nghị định: Quy định chi tiết một số điều của Luật An toàn thực phẩm, số 15/2022/NĐ-CP, ban hành ngày 28/01/2022, tại Hà Nội, bởi Chính phủ)
(*Nguồn: Điều 9 - Mục 5 - Chương I, văn bản Thông tư: Hướng dẫn kỹ thuật về phòng ngừa, ứng phó sự cố chất thải, số 2025/TT-BNNMT, ban hành ngày 14/07/2025, tại Hà Nội, bởi Bộ Nông nghiệp và môi trường)
(*Nguồn: Điều 1, Mục 4, văn bản Kế hoạch: Xây dựng chính sách năm 2014, số 32/KH/CP, ban hành ngày 21/06/2024 tại Hà Nội, bởi Chính phủ)
- Bắt buộc phải trích dẫn theo đúng định dạng trong mẫu.
- Thêm tất cả những phần có trong Metadata theo đúng cấu trúc, phần nào không có thì bỏ qua (tương tự trong các ví dụ mẫu).
- Phải tìm và trích dẫn nguồn từ phần Metadata sử dụng để trả lời với mỗi ý.
- Metadata được cung cấp ở cuối mỗi tài liệu, vì vậy sẽ luôn có nguồn.

### 4. Tuân thủ nội dung:
- Chỉ sử dụng thông tin từ nội dung được cung cấp trong phần 'Tài liệu tham khảo:' để trả lời câu hỏi, có thể suy diễn và suy luận từ thông tin. Nhưng tuyệt đối không thêm thông tin từ bên ngoài.

### 5. Xử lý trường hợp không có thông tin:
- Nêu không có thông tin phù hợp trong dữ liệu cung cấp, bắt buộc phải trả lời như sau: 'Tôi không tìm thấy thông tin trong tài liệu'.

### 6. Phong cách trình bày:
- Không quá dài, rõ ràng nhưng vẫn đảm bảo đủ độ chi tiết, đủ ý và thông tin.
- Chuyên nghiệp và chính xác theo ngôn ngữ pháp luật Việt Nam.
- Tránh sử dụng ngôn ngữ không trang trọng.

### 7. Đảm bảo chất lượng:
- Mọi câu trả lời cần được kiểm tra để đảm bảo tối đa tính chính xác và rõ ràng trước khi gửi đi.

Lưu ý quan trọng: Tất cả các những yêu cầu trên chỉ dùng để hướng dẫn những quy định khi bạn trả lời câu hỏi, khi trả lời bạn chỉ cần đảm bảo các yêu cầu trên, bắt buộc khi trả lời chỉ thực hiện trả lời theo yêu cầu và tuyệt đối không được tạo ra các yêu cầu ### phía trên.
"""

def create_prompt_template(system_prompt):
  prompt_template = ChatPromptTemplate.from_messages([
      ("system", system_prompt),
      ("human", """
      Dưới đây là câu hỏi bạn cần trả lời:
      {question}

      Tài liệu tham khảo:
      {context}
      """)
  ])

  return prompt_template


def get_response(prompt_template, llm, question, context):
  chain = prompt_template | llm
  answer = chain.invoke({"question": question, "context": context})

  return answer.content

In [ ]:
metadata_label_map = {
    "dieu": "Điều",
    "muc" : "Mục",
    "chuong": "Chương",
    "loai_van_ban": "Loại văn bản",
    "chu_de": "Chủ đề",
    "ma_so": "Mã số",
    "ngay_ban_hanh": "Ngày ban hành",
    "noi_ban_hanh": "Nơi ban hành",
    "co_quan_ban_hanh": "Cơ quan ban hành",
    "can_cu" : "Căn cứ"
}

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=".env")

openai_api_key = os.getenv("OPENAI_API_KEY")

llm = ChatOpenAI(
    model = 'gpt-4o',
    openai_api_key = openai_api_key,
    temperature=0
)

In [73]:
question = "Nội dung chính của Điều 1 trong nghị quyết số 33/NQ-CP là gì?"
context = get_context(question)
prompt_template = create_prompt_template(system_prompt)
chain = prompt_template | llm
answer = get_response(prompt_template, llm, question, context)

In [74]:
print(answer)

Nội dung chính của Điều 1 trong nghị quyết số 33/NQ-CP không được cung cấp trong tài liệu tham khảo. Tôi không tìm thấy thông tin trong tài liệu.
